In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
%pip install highway-env
%pip install stable-baselines3
%pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.7/107.7 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.2/187.2 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 404.7/404.7 kB 25.6 MB/s eta 0:00:00


In [ ]:
%cd /content/drive/MyDrive/cs-272-final-project/custom/best-model

/content/drive/MyDrive/cs-272-final-project/custom/best-model


In [ ]:
# Ignore warnings
import warnings
warnings.filterwarnings("ignore",category=DeprecationWarning,)
warnings.filterwarnings("ignore", message=".*Kernel._parent_header.*")

In [ ]:
import os, glob, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import gymnasium as gym
from custom_env import AccidentEnv
import highway_env  # registers env ids
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env

try:
    gym.register(id="MyCustomEnv-v0", entry_point="custom_env:AccidentEnv")
except gym.error.RegistrationError:
    print("Env already registered")
    pass  # already registered (e.g., on reruns)
except Exception as e:
    print(e)

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
# optuna_ppo_highway.py
import optuna
from optuna.exceptions import TrialPruned

# ------------------------------ Config ------------------------------
SEED = 42
ENV_ID = "MyCustomEnv-v0"


# Folder layout (run this from highway/hyperparam-tuned or any folder you want)
# BASE_DIR     = os.getcwd()
# RUNS_ROOT    = os.path.join(BASE_DIR, "runs_monitor")     # per-trial monitor logs
# FINAL_ROOT   = os.path.join(RUNS_ROOT, "final_best")      # final training logs
# MODELS_DIR   = os.path.join(BASE_DIR, "models")
# PLOTS_DIR    = os.path.join(BASE_DIR, "plots")
# os.makedirs(MODELS_DIR, exist_ok=True)
# os.makedirs(PLOTS_DIR, exist_ok=True)

# ------------------------------ Helpers ------------------------------

def make_train_env(monitor_dir, n_envs=4):
    os.makedirs(monitor_dir, exist_ok=True)
    return make_vec_env(
        ENV_ID,
        n_envs=n_envs,
        seed=SEED,
        monitor_dir=monitor_dir,   # logs training episodes to CSVs
    )

# --- Plot 1: learning curve from Monitor CSVs (episodes on X) ---
def plot_learning_curve_monitor(monitor_dir: str, out_png: str, smooth_window: int = 50, title="Learning Curve — Custom Env PPO (tuned)"):
    files = sorted(glob.glob(os.path.join(monitor_dir, "**", "*.csv"), recursive=True))
    if not files:
        raise FileNotFoundError(f"No monitor CSVs under: {monitor_dir}")

    dfs = []
    for f in files:
        try:
            df = pd.read_csv(f, comment="#")
            if "r" in df.columns:
                dfs.append(df[["r"]].copy())
        except Exception:
            pass
    if not dfs:
        raise RuntimeError("Found CSVs but no episodic returns (column 'r')")

    log = pd.concat(dfs, ignore_index=True)
    log["episode"] = np.arange(1, len(log) + 1)
    win = max(1, min(smooth_window, len(log)))
    log["r_smooth"] = log["r"].rolling(win, min_periods=1).mean()

    os.makedirs(os.path.dirname(out_png) or ".", exist_ok=True)
    plt.figure()
    plt.plot(log["episode"], log["r_smooth"])
    plt.xlabel("Training Episodes")
    plt.ylabel("Mean Episodic Return (smoothed)")
    plt.title(title)
    plt.tight_layout()
    plt.savefig(out_png, dpi=140)
    plt.close()
    print(f"[plot] saved: {out_png}")

# --- Plot 2: 500-episode deterministic evaluation violin ---
def evaluate_and_plot_violin(model_path: str, out_png: str,
                             env_id: str = ENV_ID,
                             n_episodes: int = 500,
                             deterministic: bool = True,
                             seed: int = SEED):
    env = gym.make(env_id)
    model = PPO.load(model_path)

    returns = []
    for _ in range(n_episodes):
        obs, info = env.reset(seed=seed)
        done = False
        ep_ret = 0.0
        while not done:
            action, _ = model.predict(obs, deterministic=deterministic)  # no exploration
            obs, reward, terminated, truncated, info = env.step(action)
            ep_ret += reward
            done = terminated or truncated
        returns.append(ep_ret)
    env.close()

    os.makedirs(os.path.dirname(out_png) or ".", exist_ok=True)
    plt.figure()
    plt.violinplot([returns], showmeans=True)
    plt.xticks([1], ["PPO"])
    plt.ylabel(f"Episodic Return (n={n_episodes}, deterministic)")
    plt.title("Performance Test — Custom Env PPO (tuned)")
    plt.tight_layout()
    plt.savefig(out_png, dpi=140)
    plt.close()
    print(f"[plot] saved: {out_png}")

# --- Metric from Monitor CSVs: mean of last-K episode returns ---
def last_k_mean(monitor_dir: str, k: int = 200) -> float:
    files = sorted(glob.glob(os.path.join(monitor_dir, "**", "*.csv"), recursive=True))
    vals = []
    for f in files:
        try:
            df = pd.read_csv(f, comment="#")
            if "r" in df.columns:
                vals.extend(df["r"].tolist())
        except Exception:
            pass
    if not vals:
        return -1e9
    vals = vals[-k:] if len(vals) >= k else vals
    return float(np.mean(vals))

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
# ------------------------------ Optuna objective ------------------------------
def objective(trial: optuna.trial.Trial) -> float:
    # Search space (solid defaults for PPO)
    lr         = trial.suggest_float("lr", 1e-5, 3e-3, log=True)
    n_steps    = trial.suggest_categorical("n_steps", [1024, 2048])
    batch_size = trial.suggest_categorical("batch_size", [64, 128, 256])
    clip       = trial.suggest_float("clip_range", 0.1, 0.3)
    gae_lam    = trial.suggest_float("gae_lambda", 0.90, 0.98)
    gamma      = trial.suggest_float("gamma", 0.97, 0.999)
    ent_coef   = trial.suggest_float("ent_coef", 1e-4, 1e-2, log=True)
    n_epochs   = trial.suggest_categorical("n_epochs", [5, 10])

    trial_dir = os.path.join(RUNS_ROOT, f"trial_{trial.number:03d}")
    env = make_train_env(trial_dir, n_envs=4)

    model = PPO(
        "MlpPolicy",
        env,
        verbose=0,
        seed=SEED,
        learning_rate=lr,
        n_steps=n_steps,
        batch_size=batch_size,
        clip_range=clip,
        gae_lambda=gae_lam,
        gamma=gamma,
        ent_coef=ent_coef,
        n_epochs=n_epochs,
    )

    # ---- run fixed number of PPO updates; prune early if underperforming ----
    n_updates = 8                                # small, fast trials
    steps_per_update = n_steps * 4               # n_envs = 4
    for u in range(n_updates):
        model.learn(total_timesteps=steps_per_update,
                    reset_num_timesteps=False, progress_bar=False)

        if u >= 3:  # warmup a few updates before judging
            score = last_k_mean(trial_dir, k=100)
            trial.report(score, step=u)
            if trial.should_prune():
                env.close()
                raise TrialPruned()

    env.close()

    # Score = mean last-K ep returns from Monitor CSVs
    return last_k_mean(trial_dir, k=200)

In [ ]:
def main():
    os.makedirs(RUNS_ROOT, exist_ok=True)

    # 1) Hyperparameter search
    sampler = optuna.samplers.TPESampler(seed=SEED, n_startup_trials=5)
    pruner  = optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=3)

    study = optuna.create_study(direction="maximize", sampler=sampler, pruner=pruner)
    study.optimize(objective, n_trials=16, timeout=7*60*60)

    print("Best trial:", study.best_trial.number)
    print("Best value (last-k mean):", study.best_value)
    print("Best params:", study.best_params)

    with open(os.path.join(MODELS_DIR, "ppo_best_params_custom.json"), "w") as f:
        json.dump(study.best_params, f, indent=2)

    # 2) Final training with best params (longer budget)
    os.makedirs(FINAL_ROOT, exist_ok=True)
    env = make_train_env(FINAL_ROOT, n_envs=4)

    bp = study.best_params
    model = PPO(
        "MlpPolicy",
        env,
        verbose=1,
        seed=SEED,
        learning_rate=bp["lr"],
        n_steps=bp["n_steps"],
        batch_size=bp["batch_size"],
        clip_range=bp["clip_range"],
        gae_lambda=bp["gae_lambda"],
        gamma=bp["gamma"],
        ent_coef=bp["ent_coef"],
        n_epochs=bp["n_epochs"],
    )

    total_timesteps = bp["n_steps"] * 4 * 40  # ~40 updates
    model.learn(total_timesteps=total_timesteps, progress_bar=False)
    env.close()

    final_model_path = os.path.join(MODELS_DIR, "ppo_custom_env_tuned")
    model.save(final_model_path)
    print(f"[model] saved: {final_model_path}.zip")

    # 3) Plots
    plot_learning_curve_monitor(
        FINAL_ROOT,
        os.path.join(PLOTS_DIR, "learning_custom_env_tuned.png"),
    )

    evaluate_and_plot_violin(
        final_model_path + ".zip",
        os.path.join(PLOTS_DIR, "violin_custom_env_tuned.png"),
        env_id=ENV_ID,
        n_episodes=500,
        deterministic=True,
    )

if __name__ == "__main__":
    main()

[I 2025-12-01 05:59:02,874] A new study created in memory with name: no-name-9f126485-2b39-4855-8885-a2670ad12639
[I 2025-12-01 06:37:46,281] Trial 0 finished with value: 10.672395815 and parameters: {'lr': 8.468008575248323e-05, 'n_steps': 1024, 'batch_size': 64, 'clip_range': 0.1116167224336399, 'gae_lambda': 0.9692940916619948, 'gamma': 0.9874323353405531, 'ent_coef': 0.0026070247583707684, 'n_epochs': 10}. Best is trial 0 with value: 10.672395815.
[I 2025-12-01 07:15:21,595] Trial 1 finished with value: 12.109164114999999 and parameters: {'lr': 0.0011536162338241388, 'n_steps': 1024, 'batch_size': 256, 'clip_range': 0.18638900372842315, 'gae_lambda': 0.9232983312158434, 'gamma': 0.987743733946949, 'ent_coef': 0.00019010245319870352, 'n_epochs': 10}. Best is trial 1 with value: 12.109164114999999.
[I 2025-12-01 07:53:20,006] Trial 2 finished with value: 12.134028739999998 and parameters: {'lr': 0.0001348157560360141, 'n_steps': 1024, 'batch_size': 128, 'clip_range': 0.22150897038028

Best trial: 5
Best value (last-k mean): 14.14842115
Best params: {'lr': 0.0009613391708676594, 'n_steps': 2048, 'batch_size': 128, 'clip_range': 0.28645806838235743, 'gae_lambda': 0.9512070801974045, 'gamma': 0.996322279063193, 'ent_coef': 0.00016056796586828547, 'n_epochs': 5}
Using cpu device
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 9.94     |
|    ep_rew_mean     | 7.93     |
| time/              |          |
|    fps             | 14       |
|    iterations      | 1        |
|    time_elapsed    | 566      |
|    total_timesteps | 8192     |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 11.3        |
|    ep_rew_mean          | 9.22        |
| time/                   |             |
|    fps                  | 14          |
|    iterations           | 2           |
|    time_elapsed         | 1132        |
|    total_timesteps      | 

In [ ]:
# Training the best params from above + n_steps = 4096
def main():
    BASE_DIR     = os.getcwd()
    MODELS_DIR   = os.path.join(BASE_DIR, "models")
    PLOTS_DIR    = os.path.join(BASE_DIR, "plots")
    os.makedirs(MODELS_DIR, exist_ok=True)
    os.makedirs(PLOTS_DIR, exist_ok=True)

    # === Log training episodes via Monitor CSVs ===
    MONITOR_DIR = os.path.join(BASE_DIR, "runs_monitor", "4096_tuned")
    os.makedirs(MONITOR_DIR, exist_ok=True)

    # Vectorized training env with monitor logging
    vec_env = make_vec_env(
        "MyCustomEnv-v0",
        n_envs=4,
        seed=42,
        monitor_dir=MONITOR_DIR,   # <-- this logs training episodes
    )

    model = PPO(
        "MlpPolicy",
        vec_env,
        verbose=1,
        seed=42,
        learning_rate=9.61e-4,
        n_steps=4096,
        batch_size=128,
        clip_range=0.286,
        gae_lambda=0.951,
        gamma=0.9963,
        ent_coef=1.6e-4,
        n_epochs=5,
    )

    total_timesteps = 409600 #4096 * 4 * 40  # ~40 updates
    model.learn(total_timesteps=total_timesteps, progress_bar=False)
    vec_env.close()

    final_model_path = os.path.join(MODELS_DIR, "ppo_custom_n_steps=4096_tuned")
    model.save(final_model_path)
    print(f"[model] saved: {final_model_path}.zip")

    plot_learning_curve_monitor(MONITOR_DIR, os.path.join(PLOTS_DIR, "ppo_custom_n_steps=4096_tuned_learning.png"),
                            title="Learning Curve — PPO TUNED(Custom Env)")
    evaluate_and_plot_violin(final_model_path, os.path.join(PLOTS_DIR, "ppo_custom_n_steps=4096_tuned_violin.png"))

if __name__ == "__main__":
    main()


Using cpu device
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 10       |
|    ep_rew_mean     | 8.13     |
| time/              |          |
|    fps             | 14       |
|    iterations      | 1        |
|    time_elapsed    | 1098     |
|    total_timesteps | 16384    |
---------------------------------


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 11.4        |
|    ep_rew_mean          | 9.29        |
| time/                   |             |
|    fps                  | 14          |
|    iterations           | 2           |
|    time_elapsed         | 2187        |
|    total_timesteps      | 32768       |
| train/                  |             |
|    approx_kl            | 0.029099066 |
|    clip_fraction        | 0.206       |
|    clip_range           | 0.286       |
|    entropy_loss         | -1.58       |
|    explained_variance   | 0.00214     |
|    learning_rate        | 0.000961    |
|    loss                 | 2.73        |
|    n_updates            | 5           |
|    policy_gradient_loss | -0.0329     |
|    value_loss           | 7.16        |
-----------------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 13    

NameError: name 'plot_learning_curve_monitor' is not defined

In [ ]:
BASE_DIR  = os.getcwd()
MODEL_ZIP = os.path.join(BASE_DIR, "models", "ppo_custom_n_steps=4096_tuned.zip")
PLOTS_DIR = os.path.join(BASE_DIR, "plots")
os.makedirs(PLOTS_DIR, exist_ok=True)

# === Log training episodes via Monitor CSVs ===
MONITOR_DIR = os.path.join(BASE_DIR, "runs_monitor", "4096_tuned")
#os.makedirs(MONITOR_DIR, exist_ok=True)

plot_learning_curve_monitor(MONITOR_DIR, os.path.join(PLOTS_DIR, "ppo_custom_n_steps=4096_tuned_learning.png"),
                            title="Learning Curve — PPO TUNED(Custom Env)")
evaluate_and_plot_violin(MODEL_ZIP, os.path.join(PLOTS_DIR, "ppo_custom_n_steps=4096_tuned_violin.png"))

[plot] saved: /content/drive/MyDrive/cs-272-final-project/custom/best-model/plots/ppo_custom_n_steps=4096_tuned_learning.png


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


[plot] saved: /content/drive/MyDrive/cs-272-final-project/custom/best-model/plots/ppo_custom_n_steps=4096_tuned_violin.png


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
# Training the best params + adding linear scheduling + vecnormalise
from stable_baselines3.common.vec_env import VecNormalize

def linear(v0): return lambda p: v0 * p  # p: 1→0

def main():
    BASE_DIR     = os.getcwd()
    MODELS_DIR   = os.path.join(BASE_DIR, "models")
    PLOTS_DIR    = os.path.join(BASE_DIR, "plots")
    os.makedirs(MODELS_DIR, exist_ok=True)
    os.makedirs(PLOTS_DIR, exist_ok=True)

    # === Log training episodes via Monitor CSVs ===
    MONITOR_DIR = os.path.join(BASE_DIR, "runs_monitor", "manual_tuned")
    os.makedirs(MONITOR_DIR, exist_ok=True)

    # Vectorized training env with monitor logging
    vec_env = make_vec_env(
        "MyCustomEnv-v0",
        n_envs=4,
        seed=42,
        monitor_dir=MONITOR_DIR,   # <-- this logs training episodes
    )
    vec_env = VecNormalize(vec_env, norm_obs=True, norm_reward=True, clip_obs=10.0)

    model = PPO(
        "MlpPolicy",
        vec_env,
        verbose=1,
        seed=42,
        learning_rate=linear(1e-3),
        n_steps=4096,
        batch_size=256,
        clip_range=linear(0.25),
        gae_lambda=0.96,
        gamma=0.995,
        ent_coef=1e-3,
        n_epochs=10,
    )

    total_timesteps = 204800 #4096 * 4 * 40  # ~40 updates
    model.learn(total_timesteps=total_timesteps, progress_bar=False)

    vecnorm_path = os.path.join(MODELS_DIR, "vecnorm_manual_tuned.pkl")
    vec_env.save(vecnorm_path)
    vec_env.close()

    final_model_path = os.path.join(MODELS_DIR, "ppo_custom_manual_tuned")
    model.save(final_model_path)
    print(f"[model] saved: {final_model_path}.zip")

    plot_learning_curve_monitor(MONITOR_DIR, os.path.join(PLOTS_DIR, "ppo_custom_manual_tuned_learning.png"),
                            title="Learning Curve — PPO MANUAL TUNED(Custom Env)")
    evaluate_and_plot_violin(final_model_path, os.path.join(PLOTS_DIR, "ppo_custom_manual_tuned_violin.png"))

if __name__ == "__main__":
    main()


Using cpu device
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 9.63     |
|    ep_rew_mean     | 7.77     |
| time/              |          |
|    fps             | 15       |
|    iterations      | 1        |
|    time_elapsed    | 1025     |
|    total_timesteps | 16384    |
---------------------------------


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


----------------------------------------
| rollout/                |            |
|    ep_len_mean          | 11.1       |
|    ep_rew_mean          | 9.04       |
| time/                   |            |
|    fps                  | 15         |
|    iterations           | 2          |
|    time_elapsed         | 2054       |
|    total_timesteps      | 32768      |
| train/                  |            |
|    approx_kl            | 0.02249533 |
|    clip_fraction        | 0.237      |
|    clip_range           | 0.23       |
|    entropy_loss         | -1.59      |
|    explained_variance   | -0.561     |
|    learning_rate        | 0.00092    |
|    loss                 | 0.242      |
|    n_updates            | 10         |
|    policy_gradient_loss | -0.0257    |
|    value_loss           | 0.715      |
----------------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 11.9        |
|    ep_rew_m

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


[plot] saved: /content/drive/MyDrive/cs-272-final-project/custom/best-model/plots/ppo_custom_manual_tuned_learning.png
[plot] saved: /content/drive/MyDrive/cs-272-final-project/custom/best-model/plots/ppo_custom_manual_tuned_violin.png


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
